# HPIE QLoRA Fine-Tuning
**Model:** Llama 3.1 8B Instruct  
**Method:** QLoRA (4-bit quantisation + LoRA adapters)  
**GPU:** T4 16GB (Colab Pro) or A100  
**Time:** ~45 min on T4  

## Steps
1. Install dependencies
2. Upload your `hpie_train.jsonl` and `hpie_eval.jsonl`
3. Run fine-tuning
4. Download the adapter


In [ ]:
# Step 1 — Install dependencies
!pip install -q transformers peft bitsandbytes accelerate datasets trl huggingface_hub

In [ ]:
# Step 2 — Login to Hugging Face (needed to access Llama 3.1)
from huggingface_hub import login
login()  # paste your HF token when prompted
# Get token from: https://huggingface.co/settings/tokens
# You also need to accept Llama 3.1 license at:
# https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

In [ ]:
# Step 3 — Upload your training files
from google.colab import files
print('Upload hpie_train.jsonl and hpie_eval.jsonl')
uploaded = files.upload()

In [ ]:
# Step 4 — Verify GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Step 5 — Load training data
import json
from datasets import Dataset

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def format_prompt(ex):
    return (
        f'### Instruction:\n{ex["instruction"]}\n\n'
        f'### Input:\n{ex["input"]}\n\n'
        f'### Response:\n{ex["output"]}'
    )

train_records = load_jsonl('hpie_train.jsonl')
eval_records  = load_jsonl('hpie_eval.jsonl')

train_dataset = Dataset.from_list([{'text': format_prompt(r)} for r in train_records])
eval_dataset  = Dataset.from_list([{'text': format_prompt(r)} for r in eval_records])

print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')
print('\nSample:')
print(train_dataset[0]['text'][:400])

In [ ]:
# Step 6 — Load Llama 3.1 8B in 4-bit
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL_NAME = 'meta-llama/Meta-Llama-3.1-8B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
model = prepare_model_for_kbit_training(model)
print('Base model loaded in 4-bit.')

In [ ]:
# Step 7 — Apply LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 8 — Fine-tune
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='./hpie-llama-qlora',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim='paged_adamw_8bit',
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    max_seq_length=2048,
    dataset_text_field='text',
    packing=False,
)

trainer.train()

In [ ]:
# Step 9 — Save and download adapter
import shutil

model.save_pretrained('./hpie-llama-qlora')
tokenizer.save_pretrained('./hpie-llama-qlora')

shutil.make_archive('hpie-llama-qlora', 'zip', '.', 'hpie-llama-qlora')
files.download('hpie-llama-qlora.zip')
print('Adapter downloaded. Unzip into your HPIE project folder.')